till now we have done many experiment on GCN and got an accuracy of aroung 80% for node classification and 0.5 approx ari for the optimal model
here we are going to implement shallow node embeddings like deepwalk and node2vec which we learned and experiment with things and compare them with spetral clustering because both of them will try to build the embeddings of nodes without using any features of graph but the spectral clustering using global view like entire laplacian matrix but the shallow node embeddings just go use local neighbour hood of graph using random walks

we already implemented this on SBM and got to know each one will perform well on their respective situations like spectral clustering does well when the community structure is good and visible globally but comuputationally expensive and this shallow node embeddings are less prone to noise and efficient in computation.

As we know the concept of spectral clustering and shallow node embeddings from notes and ppt we will directly dive into implementation and learn new things

In [19]:
import os
import sys

# 1. Move up one level from the 'notebooks' folder to the root project directory
root_path = os.path.abspath(os.path.join(".."))

# 2. Point directly to your 'src' folder
src_path = os.path.join(root_path, "src")

# 3. Inject the src path into sys.path so Python knows where to find your files
if src_path not in sys.path:
    sys.path.append(src_path)

from structural_baselines_11 import generate_biased_walks,extract_skipgram_pairs,ShallowEmbeddingModel,compute_spectral_embeddings
print("imports successfully")

imports successfully


In [20]:
import torch
from torch_geometric.datasets import Planetoid

# 1. Automatically detect if an NVIDIA GPU/CUDA backend is available
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

dataset = Planetoid(root='../data/Cora', name='Cora')
data = dataset[0].to(device)

print("\n--- Cora Dataset Properties Successfully Initialized ---")
print(f"Number of distinct node features: {dataset.num_features}")
print(f"Number of target class categories: {dataset.num_classes}")
print(f"Total graph nodes loaded inside 'data': {data.num_nodes}")


--- Cora Dataset Properties Successfully Initialized ---
Number of distinct node features: 1433
Number of target class categories: 7
Total graph nodes loaded inside 'data': 2708


In [21]:
import pandas as pd
from sklearn.cluster import KMeans
from sklearn.metrics import adjusted_rand_score, normalized_mutual_info_score
import torch
import torch.nn.functional as F 
structural_summary = {
    "Pure Topological Strategy": ["Spectral Clustering", "DeepWalk (Shallow)", "Node2Vec (BFS)","Node2Vec(DFS)"],
    "ARI Score": [],
    "NMI Score": []
}

true_labels = data.y.cpu().numpy()

In [22]:
#now we will try to run spectral clustering algorithm based on the algorithm which we written in .py file
spectral_vecs = compute_spectral_embeddings(data.edge_index, data.num_nodes, num_components=7)
spectral_clusters = KMeans(n_clusters=7, random_state=42, n_init=10).fit_predict(spectral_vecs)
#what we got from that function is [num_nodes,7] matrix ignoring the 1st vector and getting till 8 vectors
structural_summary["ARI Score"].append(f"{adjusted_rand_score(true_labels, spectral_clusters):.4f}")
structural_summary["NMI Score"].append(f"{normalized_mutual_info_score(true_labels, spectral_clusters):.4f}")


if we see the parameters of function like number of walk per node and walk lenght for the cora dataset the total walks can be generated are 2708*5=13540 walks are possible 

Then no of pairs generated per walk are if u check index wise then it will be 34 so total pairs for all walks generated is 13540*34=460360 pairs

As in the function we given batch size as 4096 and our total pairs are 460360 so we divide them 460360/4096 is approx 112 so there will be 112 pairs per batch and each batch will be trained for 10 epochs .

In [23]:
#now we will write a helper function which will train shallow node embeddings
def run_shallow_pipeline(p, q, label):
    print(f"\nTraining {label} Engine...")
    walks = generate_biased_walks(data.edge_index, data.num_nodes, walk_length=10, walks_per_node=5, p=p, q=q)
    targets, contexts = extract_skipgram_pairs(walks, window_size=2)
    
    model = ShallowEmbeddingModel(num_nodes=data.num_nodes, embedding_dim=32).to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=0.02)
    
    targets, contexts = targets.to(device), contexts.to(device)
    model.train()
    batch_size = 4096
    
    for epoch in range(1, 11):
        perm = torch.randperm(targets.size(0))
        for i in range(0, targets.size(0), batch_size):
            optimizer.zero_grad()
            idx = perm[i:i+batch_size]
            b_t, b_c = targets[idx], contexts[idx]
            
            pos_loss = F.binary_cross_entropy_with_logits(model(b_t, b_c), torch.ones_like(model(b_t, b_c)))
            neg_c = torch.randint(0, data.num_nodes, b_c.shape, device=device)
            neg_loss = F.binary_cross_entropy_with_logits(model(b_t, neg_c), torch.zeros_like(model(b_t, neg_c)))
            
            (pos_loss + neg_loss).backward()
            optimizer.step()
            
    model.eval()
    vecs = model.w_target.weight.data.cpu().numpy()
    clusters = KMeans(n_clusters=7, random_state=42, n_init=10).fit_predict(vecs)
    
    structural_summary["ARI Score"].append(f"{adjusted_rand_score(true_labels, clusters):.4f}")
    structural_summary["NMI Score"].append(f"{normalized_mutual_info_score(true_labels, clusters):.4f}")

In [24]:
#calling the above function for both deepwalk and node2vec
run_shallow_pipeline(p=1.0, q=1.0, label="DeepWalk")


Training DeepWalk Engine...


In [25]:
run_shallow_pipeline(p=0.5, q=2.0, label="Node2Vec(BFS)")
#this is a biased random walk and explore the local sructure more like BFS traversal


Training Node2Vec(BFS) Engine...


In [26]:
run_shallow_pipeline(p=2.0, q=0.5, label="Node2Vec(DFS)")


Training Node2Vec(DFS) Engine...


In [27]:
#trying to display the results
print("results of community detection using only graph structure")
print(pd.DataFrame(structural_summary).to_string(index=False))


results of community detection using only graph structure
Pure Topological Strategy ARI Score NMI Score
      Spectral Clustering    0.0034    0.0914
       DeepWalk (Shallow)    0.2488    0.3452
           Node2Vec (BFS)    0.2173    0.3092
            Node2Vec(DFS)    0.2519    0.3358


based on the results we can say that spectral clustering not did community detection it just randomly assigned the nodes thats why its ARI gt reduced to below 0 and deepwalk and node2vec are somewhat good as they use local neighborhood of the node and the homophile of the this cora dataset is good which is aroung 80% so that is the reason which they performed good 

The reason for this bad result for spectral clustering is unormalized laplacian matrix usage where it will just focus on gaint hubs for nodes which more connections are present rather than actaul community structure in that way it will ignore the small isolated nodes

This is what happens to each entry when we multiply D^-1/2 to both sides of adjancy matrix
1. The Matrix Breakdown (Element-by-Element)When you look at a single specific link between Node $i$ and Node $j$ in the raw adjacency matrix $A$, the value is simply $1$ if they are connected, or $0$ if they aren't.When we apply $D^{-1/2}$ to both sides, the new, normalized value for that exact link changes to a fraction:$$\text{Normalized Link Value} = \frac{A_{ij}}{\sqrt{\text{Degree}(i)} \times \sqrt{\text{Degree}(j)}}$$Instead of every edge having an identical weight of $1$, the weight of the connection is now inversely proportional to the popularity of both nodes.

In [28]:
#now we will try with noralized lalacian matrix of spectral clustering and observe the results
#for that we are going to use symmetric normalization where it will give equal importance to all nodes irrespective of how many connections they have
import os
import sys

# 1. Move up one level from the 'notebooks' folder to the root project directory
root_path = os.path.abspath(os.path.join(".."))

# 2. Point directly to your 'src' folder
src_path = os.path.join(root_path, "src")

# 3. Inject the src path into sys.path so Python knows where to find your files
if src_path not in sys.path:
    sys.path.append(src_path)

from structural_baselines_11 import compute_spectral_embeddings_normalized
print("imports successfully")


imports successfully


In [29]:
normalized_spectral_results = {
    "Topological Strategy": ["Spectral Clustering (Symmetrically Normalized)"],
    "ARI Score": [],
    "NMI Score": []
}

# Pull ground-truth categories from data object
true_labels = data.y.cpu().numpy()

In [30]:
#calling the normalized spectral clustering function 
norm_vecs = compute_spectral_embeddings_normalized(data.edge_index, data.num_nodes, num_components=7)
#doing k means clustering for community detection
norm_clusters = KMeans(n_clusters=7, random_state=42, n_init=10).fit_predict(norm_vecs)
#calculating the matrices
ari_score = adjusted_rand_score(true_labels, norm_clusters)
nmi_score = normalized_mutual_info_score(true_labels, norm_clusters)
normalized_spectral_results["ARI Score"].append(f"{ari_score:.4f}")
normalized_spectral_results["NMI Score"].append(f"{nmi_score:.4f}")

print("Nomarlized spectral clustering algorithm")
df_norm = pd.DataFrame(normalized_spectral_results)
print(df_norm.to_string(index=False))


Nomarlized spectral clustering algorithm
                          Topological Strategy ARI Score NMI Score
Spectral Clustering (Symmetrically Normalized)   -0.0094    0.0265


In [31]:
structural_summary = {
    "Pure Topological Strategy": ["DeepWalk (Shallow)", "Node2Vec (BFS)","Node2Vec(DFS)"],
    "ARI Score": [],
    "NMI Score": []
}
#here we are again trying to build the mebeddings of node from node2vec and deepwalk by changing the parameters like increasing the
#no of walks per node and walk length and we will see what will happen
def run_shallow_pipeline(p, q, label):
    print(f"\nTraining {label} Engine...")
    walks = generate_biased_walks(data.edge_index, data.num_nodes, walk_length=15, walks_per_node=15, p=p, q=q)
    targets, contexts = extract_skipgram_pairs(walks, window_size=2)
    
    model = ShallowEmbeddingModel(num_nodes=data.num_nodes, embedding_dim=32).to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=0.02)
    
    targets, contexts = targets.to(device), contexts.to(device)
    model.train()
    batch_size = 4096
    
    for epoch in range(1, 11):
        perm = torch.randperm(targets.size(0))
        for i in range(0, targets.size(0), batch_size):
            optimizer.zero_grad()
            idx = perm[i:i+batch_size]
            b_t, b_c = targets[idx], contexts[idx]
            
            pos_loss = F.binary_cross_entropy_with_logits(model(b_t, b_c), torch.ones_like(model(b_t, b_c)))
            neg_c = torch.randint(0, data.num_nodes, b_c.shape, device=device)
            neg_loss = F.binary_cross_entropy_with_logits(model(b_t, neg_c), torch.zeros_like(model(b_t, neg_c)))
            
            (pos_loss + neg_loss).backward()
            optimizer.step()
            
    model.eval()
    vecs = model.w_target.weight.data.cpu().numpy()
    clusters = KMeans(n_clusters=7, random_state=42, n_init=10).fit_predict(vecs)
    
    structural_summary["ARI Score"].append(f"{adjusted_rand_score(true_labels, clusters):.4f}")
    structural_summary["NMI Score"].append(f"{normalized_mutual_info_score(true_labels, clusters):.4f}")


In [32]:
run_shallow_pipeline(p=1.0, q=1.0, label="DeepWalk")


Training DeepWalk Engine...


In [33]:
run_shallow_pipeline(p=0.5, q=2, label="Node2Vec (BFS)")


Training Node2Vec (BFS) Engine...


In [34]:
run_shallow_pipeline(p=2, q=0.5, label="Node2Vec(DFS)")


Training Node2Vec(DFS) Engine...


In [35]:
print("results of community detection using only graph structure")
print(pd.DataFrame(structural_summary).to_string(index=False))

results of community detection using only graph structure
Pure Topological Strategy ARI Score NMI Score
       DeepWalk (Shallow)    0.2329    0.3052
           Node2Vec (BFS)    0.1866    0.2660
            Node2Vec(DFS)    0.1981    0.2749


these reuslts are very interesting we thaught as the no of walks and length of the walks increasing the result would improve but here not the case so because when they increases they ( i mean the walks) are going beyond the local neighborhood and also as the no of walks increases we almost explore every single path from a single node  which is reuslting in oversmoothing thats why the result got reduced 